# Steering methods across depth and tasks — oracle sweep analysis

Compares the shared-autonomy arms on the runs written by
[`oracle_sweep.sh`](../oracle_sweep.sh) (or any set of `experiment.py` runs) into
`outputs/pi05_libero_oracle_sweep/`. Each run is one *block*: a method
(FC = shared flow control, FRS = native flow reversal steering, FRS+F = flow
reversal with the reversal adapter) at one *depth* (the number of denoising
steps the operator controls: `n_guided_steps` for FC, `n_reversal_steps` for the
reversal arms) on one task, plus two policy-only anchors per task (the arm's
uninformative prompt = floor, the scene's real instruction = ceiling).

The helpers live in [`analyze.py`](analyze.py); the point of this
notebook is the **performance-vs-controlled-steps figure** with one line per method.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

for candidate in (Path.cwd(), Path.cwd() / "examples/pi05/libero_shared_autonomy/notebooks"):
    if (candidate / "analyze.py").exists():
        sys.path.insert(0, str(candidate))
        REPO_ROOT = candidate.parents[3] if candidate.name == "notebooks" else Path.cwd()
        break
import analyze  # noqa: E402

ROOT = REPO_ROOT / "outputs/pi05_libero_oracle_sweep"  # point elsewhere to compare other runs
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## Load

Every trial gets `method`, `depth`, `task_id` and `mode_expressed` (the scene
element that moved most during the trial). `operator` says who drove: `policy`
for the oracle sweep, `human` for SpaceMouse sessions — keep them apart when
both are present.

In [ ]:
trials, configs = analyze.load_runs(analyze.find_runs(ROOT))
print(
    f"{len(trials)} trials, {trials['run'].nunique()} runs, {trials['task_id'].nunique()} tasks, "
    f"operators: {sorted(trials['operator'].unique())}"
)
trials.groupby(["method", "depth"], dropna=False).size().unstack(fill_value=0)

## Success rate vs. controlled steps

The figure asked for: x = denoising steps the operator controls, y = success rate
pooled over tasks, one line per method with Wilson 95% intervals. The anchors
are dashed horizontal lines (floor: the policy alone with the uninformative
prompt; ceiling: the policy alone with the real instruction).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
curve = analyze.plot_performance_vs_depth(trials, ax, metric="success")
plt.tight_layout()
plt.show()
curve

## Did the steering select the right behaviour?

Success is two-stage in the intent-agnostic setting: the steering must first
select the intended mode of the policy's prior, then the policy must execute
it. `on_target` is whether the trial's most-moved scene element is the task's
own (taken from what the ceiling anchor moves when it succeeds), regardless of
whether the trial finished. This separates *wrong mode* from *right mode, not
finished*.

In [ ]:
tagged = analyze.with_mode_accuracy(trials)
print("task targets (from the ceiling anchor):")
for task_id, target in sorted(analyze.task_targets(trials).items()):
    desc = trials.loc[trials["task_id"] == task_id, "task_description"].iloc[0]
    print(f"  task {task_id}: {target!s:40s} {desc}")

fig, ax = plt.subplots(figsize=(8, 5))
analyze.plot_performance_vs_depth(
    tagged, ax, metric="on_target", title="expressed the task's own behaviour vs. controlled steps"
)
plt.tight_layout()
plt.show()

## Both metrics side by side, and the best depth per method

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
analyze.plot_performance_vs_depth(tagged, axes[0], metric="success", title="task success")
analyze.plot_performance_vs_depth(tagged, axes[1], metric="on_target", title="right behaviour selected")
plt.tight_layout()
plt.show()
analyze.best_depth(tagged, "success")

## Per task

Rows = task, columns = (method, depth). Tasks where every arm is at the floor
are not informative for comparing methods; tasks with a gradient are the ones
to take to human trials.

In [ ]:
per_task = analyze.per_task_table(tagged, "success")
per_task.style.format("{:.0%}", na_rep="–").background_gradient(cmap="Blues", vmin=0, vmax=1)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7), sharex=True, sharey=True)
for ax, (task_id, group) in zip(axes.ravel(), tagged.groupby("task_id"), strict=False):
    desc = group["task_description"].iloc[0]
    analyze.plot_performance_vs_depth(group, ax, metric="success", title=f"task {task_id}: {desc[:38]}")
    ax.legend(fontsize=7)
    ax.set_xlabel("controlled steps")
plt.tight_layout()
plt.show()

## How much of the intent got through

From the per-step arrays: the cosine between the executed translation and the
operator's *raw* command (the clean intent, before the corruption) versus the
*served* command (what the policy was given). A method that undoes the
corruption has `cos_raw` above `cos_served`; a method that just follows the
corrupted command has the opposite. This is a continuous measure, independent
of task success, so it stays informative even where success is at the floor.

In [ ]:
steered = tagged[tagged["method"].isin(["FC", "FRS", "FRS+F"])]
transmission = analyze.step_metrics(steered)
summary = (
    transmission.groupby(["method", "depth"])[["cos_raw", "cos_served", "commanding_frac"]].mean().round(3)
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for method, group in summary.reset_index().groupby("method", sort=False):
    ax.plot(group["depth"], group["cos_raw"], marker="o", label=f"{method}: vs raw intent")
    ax.plot(
        group["depth"],
        group["cos_served"],
        marker="x",
        linestyle=":",
        label=f"{method}: vs corrupted command",
    )
ax.set_xlabel("controlled denoising steps")
ax.set_ylabel("mean cosine(executed translation, command)")
ax.set_xticks(sorted(summary.reset_index()["depth"].unique()))
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Paired comparison at a chosen depth

Blocks on the same task share the seed, so trial *i* is the same scene reset
across methods: McNemar's exact test on the discordant pairs, pooled over
tasks, is the right test for "does the adapter help".

In [ ]:
depths = sorted(tagged["depth"].dropna().unique())
at = lambda d: tagged[tagged["depth"] == d]  # noqa: E731
pd.DataFrame(
    [
        {"depth": d, **analyze.paired_test(at(d), "method", "FRS", "FRS+F", ["task_id", "trial"])}
        for d in depths
    ]
    + [
        {"depth": d, **analyze.paired_test(at(d), "method", "FC", "FRS+F", ["task_id", "trial"])}
        for d in depths
    ]
)